# 7-Input Joystick (Switch-Only) - build123d

This notebook creates a first parametric CAD baseline for a joystick with:
- X+/X-/Y+/Y- switch actuation via tilt cams
- CW/CCW switch actuation via twist cam
- Center pushbutton plunger

All dimensions are in mm.

In [1]:
from build123d import *

# Optional: only needed if you want interactive preview in VS Code OCP viewer
try:
    from ocp_vscode import show
    HAS_SHOW = True
except Exception:
    HAS_SHOW = False

MM = 1.0

In [2]:
# Global design parameters
P = {
    'plate_xy': 100.0,
    'plate_t': 4.0,
    'body_xy': 72.0,
    'body_z': 38.0,
    'overall_h': 100.0,
    'shaft_od': 16.0,
    'shaft_id': 11.0,
    'knob_od': 36.0,
    'knob_h': 28.0,
    'tilt_deg': 12.0,
    'twist_deg': 10.0,
    'clearance_slide': 0.25,
    'clearance_rot': 0.20,
    'insert_hole_d': 3.8,   # adjust to your M2.5 insert model
    'insert_hole_depth': 5.0,
    'm25_pass_d': 2.7
}

In [3]:
def make_housing(p=P):
    plate_xy = p['plate_xy']
    plate_t = p['plate_t']
    body_xy = p['body_xy']
    body_z = p['body_z']

    with BuildPart() as housing:
        # Top plate
        with BuildSketch(Plane.XY):
            Rectangle(plate_xy, plate_xy)
        extrude(amount=plate_t)

        # Under-panel body
        with BuildSketch(Plane.XY.offset(-body_z)):
            Rectangle(body_xy, body_xy)
        extrude(amount=body_z)

        # Central shaft opening through plate/body
        with BuildSketch(Plane.XY):
            Circle((p['shaft_od'] / 2) + 1.0)
        extrude(amount=plate_t + body_z, mode=Mode.SUBTRACT, both=False)

        # Bottom service opening (leave wall thickness)
        with BuildSketch(Plane.XY.offset(-body_z + 2.4)):
            Rectangle(body_xy - 6.0, body_xy - 6.0)
        extrude(amount=body_z - 2.4, mode=Mode.SUBTRACT, both=False)

        # 8 insert bosses around perimeter for cover mounting
        for sx in (-1, 1):
            for sy in (-1, 1):
                for dx, dy in ((12, 12), (24, 24)):
                    x = sx * ((body_xy / 2) - dx)
                    y = sy * ((body_xy / 2) - dy)
                    with BuildSketch(Plane.XY.offset(-body_z + 2.4)):
                        Circle(3.0)
                    extrude(amount=6.0)
                    with BuildSketch(Plane.XY.offset(-body_z + 2.4)):
                        Circle(p['insert_hole_d'] / 2)
                    extrude(amount=p['insert_hole_depth'], mode=Mode.SUBTRACT)

    return housing.part

housing = make_housing()
housing

Part at 0x116c66530, label(), #children(0)

In [4]:
def make_tilt_carrier(p=P):
    shaft_od = p['shaft_od']

    with BuildPart() as carrier:
        # Core body
        with BuildSketch(Plane.XY):
            Circle(20.0)
        extrude(amount=12.0, both=True)

        # Through bore for shaft/twist sleeve interface
        with BuildSketch(Plane.XY):
            Circle((shaft_od / 2) + p['clearance_rot'])
        extrude(amount=20.0, both=True, mode=Mode.SUBTRACT)

        # Trunnions for orthogonal pivots
        for sx in (-1, 1):
            with BuildSketch(Plane.YZ.offset(sx * 20.0)):
                Circle(3.0)
            extrude(amount=8.0, both=True)

        for sy in (-1, 1):
            with BuildSketch(Plane.XZ.offset(sy * 20.0)):
                Circle(3.0)
            extrude(amount=8.0, both=True)

        # Four cam pads (X+/X-/Y+/Y-) as simple flats
        for ax in ((1, 0), (-1, 0), (0, 1), (0, -1)):
            x = ax[0] * 16.0
            y = ax[1] * 16.0
            with BuildSketch(Plane.XY.offset(6.0)):
                Rectangle(7.0, 4.0, align=(Align.CENTER, Align.CENTER))
            extrude(amount=2.0)
            carrier.part = carrier.part.moved(Location((x, y, 0)))

    return carrier.part

tilt_carrier = make_tilt_carrier()
tilt_carrier

Part at 0x128707ed0, label(), #children(0)

In [5]:
def make_twist_sleeve(p=P):
    shaft_od = p['shaft_od']
    shaft_id = p['shaft_id']

    with BuildPart() as sleeve:
        # Sleeve body
        with BuildSketch(Plane.XY):
            Circle((shaft_od / 2) + 2.0)
            Circle((shaft_od / 2) + p['clearance_rot'], mode=Mode.SUBTRACT)
        extrude(amount=24.0, both=True)

        # Two opposite cam lobes for CW/CCW switch actuation
        for a in (0, 180):
            with BuildSketch(Plane.XY.offset(0)):
                with Locations((14.5, 0),):
                    Rectangle(6.0, 4.0)
            extrude(amount=3.0)
            if a != 0:
                sleeve.part = sleeve.part.rotate(Axis.Z, a)

        # Plunger tunnel
        with BuildSketch(Plane.XY):
            Circle((shaft_id / 2) - 0.6)
        extrude(amount=26.0, both=True, mode=Mode.SUBTRACT)

    return sleeve.part

twist_sleeve = make_twist_sleeve()
twist_sleeve

Part at 0x128858ff0, label(), #children(0)

In [6]:
def make_knob_and_plunger(p=P):
    with BuildPart() as knob:
        with BuildSketch(Plane.XY):
            Circle(p['knob_od'] / 2)
        extrude(amount=p['knob_h'])

        # Finger contour
        fillet(knob.edges().filter_by(GeomType.CIRCLE), radius=1.2)

        # Shaft socket
        with BuildSketch(Plane.XY):
            Circle((p['shaft_od'] / 2) + p['clearance_slide'])
        extrude(amount=p['knob_h'] - 4.0, mode=Mode.SUBTRACT)

        # Plunger passage
        with BuildSketch(Plane.XY):
            Circle(4.0)
        extrude(amount=p['knob_h'], mode=Mode.SUBTRACT)

    with BuildPart() as plunger:
        with BuildSketch(Plane.XY):
            Circle(3.6)
        extrude(amount=42.0)
        # Tip for center pushbutton
        with BuildSketch(Plane.XY.offset(-2.0)):
            Circle(2.0)
        extrude(amount=2.0)

    return knob.part, plunger.part

knob, plunger = make_knob_and_plunger()
knob, plunger

(Part at 0x128859630, label(), #children(0),
 Part at 0x1288599f0, label(), #children(0))

In [7]:
# Basic visualization and STEP export
if HAS_SHOW:
    show(housing, tilt_carrier, twist_sleeve, knob, plunger)

export_step(housing, 'joystick_housing.step')
export_step(tilt_carrier, 'joystick_tilt_carrier.step')
export_step(twist_sleeve, 'joystick_twist_sleeve.step')
export_step(knob, 'joystick_knob.step')
export_step(plunger, 'joystick_plunger.step')

print('Exported baseline parts: housing, tilt carrier, twist sleeve, knob, plunger')

Using port 3939
+++++
Exported baseline parts: housing, tilt carrier, twist sleeve, knob, plunger


## Next CAD Tasks

1. Replace placeholder switch clearances with exact geometry from ModelPushbuttonJHF58.
2. Add dedicated switch bracket parts for X/Y, twist, and center pushbutton mounts.
3. Add explicit hard-stop geometry for +/-12 deg tilt and +/-10 deg twist.
4. Add M2.5 insert-ready bosses to all screw interfaces and tune hole diameters to your insert brand.
5. Split housing into top shell + bottom service cover if you want easier print orientation.

## Step 1 Applied: Exact JHF58 Switch Envelope and Brackets

These cells replace placeholder switch assumptions with dimensions taken from `ModelPushbuttonJHF58` and `ModelPushbuttonJHF58Cutout`.

Outputs:
- Parametric `JHF58` envelope values
- `xy_switch_bracket` for X/Y and twist side mounting
- `center_switch_mount` for the knob pushbutton
- STEP exports for bracket prototypes

In [8]:
# JHF58 dimensions sourced from models/model_pushbutton_jhf58.ipynb
JHF58 = {
    'body_x': 5.8,
    'body_y': 5.8,
    'body_z': 5.2,
    'shaft_len': 2.17 + 2.34,
    'leg_x': 0.6,
    'leg_y': 0.26,
    'leg_z': 3.36,
    'cutout_clearance': 0.254,
}

JHF58_LEG_LOCATIONS = [
    (-1.28, 2.25), (0.0, 2.25), (1.28, 2.25),
    (-1.28, -2.25), (0.0, -2.25), (1.28, -2.25),
]

def make_jhf58_cutout(clearance: float | None = None):
    c = JHF58['cutout_clearance'] if clearance is None else clearance
    with BuildPart() as cutout:
        with BuildSketch(Plane.XY):
            Rectangle(JHF58['body_x'] + c, JHF58['body_y'] + c)
        extrude(amount=JHF58['body_z'] + c)

        # Lead relief as a single underside slot, easier to print than 6 isolated holes.
        with BuildSketch(Plane.XY):
            Rectangle(4.8, 5.0)
        extrude(amount=-(JHF58['leg_z'] + c))

    return cutout.part

jhf58_cutout = make_jhf58_cutout()
jhf58_cutout

Part at 0x128707110, label(), #children(0)

In [9]:
def make_side_switch_bracket(
    wall_thickness: float = 2.8,
    base_thickness: float = 3.0,
    switch_clearance: float | None = None,
    ear_hole_spacing_x: float = 10.0,
):
    c = JHF58['cutout_clearance'] if switch_clearance is None else switch_clearance
    sw_x = JHF58['body_x'] + c
    sw_y = JHF58['body_y'] + c
    sw_z = JHF58['body_z'] + c

    base_x = sw_x + 8.0
    base_y = 14.0
    wall_h = sw_z + 4.0

    with BuildPart() as bracket:
        # Base plate
        Box(base_x, base_y, base_thickness, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Upright wall that captures the switch body
        with Locations((0, (base_y - wall_thickness) / 2, base_thickness)):
            Box(base_x, wall_thickness, wall_h, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Switch body cavity
        with Locations((0, (base_y - wall_thickness) / 2, base_thickness + 1.5)):
            Box(sw_x, wall_thickness + 0.4, sw_z, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Lead relief channel under the body
        with Locations((0, (base_y - wall_thickness) / 2, base_thickness - (JHF58['leg_z'] - 0.5))):
            Box(4.8, wall_thickness + 0.6, JHF58['leg_z'], mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # M2.5 pass-through holes on mounting ears
        for x in (-ear_hole_spacing_x / 2, ear_hole_spacing_x / 2):
            with Locations((x, -2.0, base_thickness / 2)):
                Cylinder(radius=P['m25_pass_d'] / 2, height=base_thickness + 1.0, mode=Mode.SUBTRACT)

    return bracket.part

def make_center_switch_mount(switch_clearance: float | None = None):
    c = JHF58['cutout_clearance'] if switch_clearance is None else switch_clearance
    sw_x = JHF58['body_x'] + c
    sw_y = JHF58['body_y'] + c

    mount_x = sw_x + 16.0
    mount_y = sw_y + 16.0
    mount_z = 3.0

    with BuildPart() as mount:
        Box(mount_x, mount_y, mount_z, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Center switch pocket
        with Locations((0, 0, 1.0)):
            Box(sw_x, sw_y, JHF58['body_z'] + 0.8, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Lead relief trench
        with Locations((0, 0, -JHF58['leg_z'] + 1.0)):
            Box(5.0, 5.2, JHF58['leg_z'] + 0.8, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Corner mount holes
        hole_dx = (mount_x / 2) - 3.5
        hole_dy = (mount_y / 2) - 3.5
        for sx in (-1, 1):
            for sy in (-1, 1):
                with Locations((sx * hole_dx, sy * hole_dy, mount_z / 2)):
                    Cylinder(radius=P['m25_pass_d'] / 2, height=mount_z + 1.0, mode=Mode.SUBTRACT)

    return mount.part

xy_switch_bracket = make_side_switch_bracket()
twist_switch_bracket = make_side_switch_bracket()
center_switch_mount = make_center_switch_mount()

if HAS_SHOW:
    show(xy_switch_bracket, twist_switch_bracket, center_switch_mount, jhf58_cutout)

export_step(xy_switch_bracket, 'joystick_xy_switch_bracket.step')
export_step(twist_switch_bracket, 'joystick_twist_switch_bracket.step')
export_step(center_switch_mount, 'joystick_center_switch_mount.step')

print('Exported: joystick_xy_switch_bracket.step, joystick_twist_switch_bracket.step, joystick_center_switch_mount.step')

+c++
Exported: joystick_xy_switch_bracket.step, joystick_twist_switch_bracket.step, joystick_center_switch_mount.step


## Step 2 Applied: Tilt and Twist Hard Stops

This step adds printable hard-stop parts:
- `tilt_stop_ring`: limits tilt motion to about +/-12 degrees with 4 stop tabs.
- `twist_stop_ring`: limits twist to about +/-10 degrees using opposed sector windows and stop ears.

Both are exported as STEP for integration with the housing/carrier/sleeve stack.

In [11]:
import math

def make_tilt_stop_ring(p=P, stop_clearance_deg: float = 0.8):
    # Target +/-12 deg effective stop with a small assembly margin.
    stop_angle = p['tilt_deg'] + stop_clearance_deg
    ring_od = 40.0
    ring_id = p['shaft_od'] + 1.0
    ring_t = 3.0
    tab_len = 4.0
    tab_w = 5.0

    with BuildPart() as ring:
        with BuildSketch(Plane.XY):
            Circle(ring_od / 2)
            Circle(ring_id / 2, mode=Mode.SUBTRACT)
        extrude(amount=ring_t)

        # Four stop tabs at cardinal directions.
        for a in (0, 90, 180, 270):
            rad = math.radians(a)
            x = (ring_od / 2 - 1.5) * math.cos(rad)
            y = (ring_od / 2 - 1.5) * math.sin(rad)
            with Locations((x, y, ring_t)):
                Box(tab_len, tab_w, 2.4, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Engraved alignment dots near +/- stop angles (robust marker geometry).
        mark_r = (ring_id + ring_od) / 4
        for sign in (-1, 1):
            a = math.radians(sign * stop_angle)
            mx = mark_r * math.cos(a)
            my = mark_r * math.sin(a)
            with Locations((mx, my, ring_t)):
                Cylinder(radius=0.8, height=0.5, mode=Mode.SUBTRACT)

    return ring.part

def make_twist_stop_ring(p=P, stop_clearance_deg: float = 0.6):
    # Target +/-10 deg effective twist stop with small assembly margin.
    stop_angle = p['twist_deg'] + stop_clearance_deg
    ring_od = 32.0
    ring_id = p['shaft_od'] + 0.8
    ring_t = 3.0

    with BuildPart() as ring:
        with BuildSketch(Plane.XY):
            Circle(ring_od / 2)
            Circle(ring_id / 2, mode=Mode.SUBTRACT)
        extrude(amount=ring_t)

        # Fixed stop ears that mate with sleeve tabs (180 deg apart).
        for a in (0, 180):
            rad = math.radians(a)
            x = (ring_od / 2 - 1.8) * math.cos(rad)
            y = (ring_od / 2 - 1.8) * math.sin(rad)
            with Locations((x, y, ring_t)):
                Box(3.0, 4.0, 2.8, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Clearance notches centered at +/-stop_angle.
        notch_r = (ring_id + ring_od) / 4
        for sign in (-1, 1):
            a = math.radians(sign * stop_angle)
            x = notch_r * math.cos(a)
            y = notch_r * math.sin(a)
            with Locations((x, y, ring_t / 2)):
                Box(2.4, 5.5, ring_t + 0.6, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.CENTER))

    return ring.part

tilt_stop_ring = make_tilt_stop_ring()
twist_stop_ring = make_twist_stop_ring()

if HAS_SHOW:
    show(tilt_stop_ring, twist_stop_ring)

export_step(tilt_stop_ring, 'joystick_tilt_stop_ring.step')
export_step(twist_stop_ring, 'joystick_twist_stop_ring.step')

print('Exported: joystick_tilt_stop_ring.step, joystick_twist_stop_ring.step')

++
Exported: joystick_tilt_stop_ring.step, joystick_twist_stop_ring.step


## Step 3 Applied: M2.5 Insert-Ready Mounting Integration

This step integrates screw-ready mounting into the core parts:
- Housing gets four internal insert bosses for stop-ring retention.
- Tilt and twist stop rings get matching M2.5 pass-through bolt patterns.

Exports are written as new STEP variants for assembly testing.

In [13]:
import math

def make_housing_with_stop_insert_bosses(p=P):
    base = make_housing(p)

    boss_radius = 3.2
    boss_height = 7.0
    boss_circle_d = 28.0  # centered around shaft axis
    boss_z = -p['body_z'] + 2.4  # near bottom service side

    with BuildPart() as enriched:
        insert(base)

        # Add four bosses and insert bores for M2.5 heat inserts.
        for i in range(4):
            a = math.radians(i * 90.0 + 45.0)
            x = (boss_circle_d / 2) * math.cos(a)
            y = (boss_circle_d / 2) * math.sin(a)

            with Locations((x, y, boss_z)):
                Cylinder(radius=boss_radius, height=boss_height, align=(Align.CENTER, Align.CENTER, Align.MIN))

            with Locations((x, y, boss_z)):
                Cylinder(
                    radius=p['insert_hole_d'] / 2,
                    height=p['insert_hole_depth'],
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER, Align.MIN),
                )

            # Through-hole from boss top for M2.5 screw shank.
            with Locations((x, y, boss_z + boss_height)):
                Cylinder(
                    radius=p['m25_pass_d'] / 2,
                    height=16.0,
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER, Align.MIN),
                )

    return enriched.part

def add_m25_bolt_pattern_to_ring(ring_part, bolt_circle_d: float, count: int = 4, p=P):
    with BuildPart() as updated:
        insert(ring_part)
        for i in range(count):
            a = math.radians((360.0 / count) * i + 45.0)
            x = (bolt_circle_d / 2) * math.cos(a)
            y = (bolt_circle_d / 2) * math.sin(a)
            with Locations((x, y, 0.0)):
                Cylinder(
                    radius=p['m25_pass_d'] / 2,
                    height=12.0,
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER, Align.MIN),
                )
    return updated.part

housing_m25 = make_housing_with_stop_insert_bosses()
tilt_stop_ring_m25 = add_m25_bolt_pattern_to_ring(make_tilt_stop_ring(), bolt_circle_d=28.0)
twist_stop_ring_m25 = add_m25_bolt_pattern_to_ring(make_twist_stop_ring(), bolt_circle_d=22.0)

if HAS_SHOW:
    show(housing_m25, tilt_stop_ring_m25, twist_stop_ring_m25)

export_step(housing_m25, 'joystick_housing_m25.step')
export_step(tilt_stop_ring_m25, 'joystick_tilt_stop_ring_m25.step')
export_step(twist_stop_ring_m25, 'joystick_twist_stop_ring_m25.step')

print('Exported: joystick_housing_m25.step, joystick_tilt_stop_ring_m25.step, joystick_twist_stop_ring_m25.step')

ccc
Exported: joystick_housing_m25.step, joystick_tilt_stop_ring_m25.step, joystick_twist_stop_ring_m25.step


## Step 4 Applied: Quick Fit-Check Assembly

This step builds a simple placement assembly for visual and numeric checks:
- Places `housing_m25`, `tilt_carrier`, `twist_sleeve`, `plunger`, `tilt_stop_ring_m25`, and `twist_stop_ring_m25`.
- Computes axis-aligned bounding-box overlap estimates.
- Reports likely clashes for fast iteration before detailed kinematic constraints are added.

In [14]:
def _bbox_overlap_mm(a, b):
    ba = a.bounding_box()
    bb = b.bounding_box()

    dx = min(ba.max.X, bb.max.X) - max(ba.min.X, bb.min.X)
    dy = min(ba.max.Y, bb.max.Y) - max(ba.min.Y, bb.min.Y)
    dz = min(ba.max.Z, bb.max.Z) - max(ba.min.Z, bb.min.Z)
    return dx, dy, dz

def _bbox_clash(a, b, tol=0.01):
    dx, dy, dz = _bbox_overlap_mm(a, b)
    return (dx > tol) and (dy > tol) and (dz > tol), (dx, dy, dz)

# Nominal placement (global Z: panel top at 0, below panel is negative).
z_housing = 0.0
z_tilt_ring = -P['body_z'] + 5.0
z_twist_ring = z_tilt_ring + 4.2
z_tilt_carrier = -10.0
z_twist_sleeve = -2.0
z_plunger = -18.0

housing_fit = housing_m25.moved(Location((0, 0, z_housing)))
tilt_ring_fit = tilt_stop_ring_m25.moved(Location((0, 0, z_tilt_ring)))
twist_ring_fit = twist_stop_ring_m25.moved(Location((0, 0, z_twist_ring)))
tilt_carrier_fit = tilt_carrier.moved(Location((0, 0, z_tilt_carrier)))
twist_sleeve_fit = twist_sleeve.moved(Location((0, 0, z_twist_sleeve)))
plunger_fit = plunger.moved(Location((0, 0, z_plunger)))

fit_pairs = [
    ('tilt_ring vs housing', tilt_ring_fit, housing_fit),
    ('twist_ring vs housing', twist_ring_fit, housing_fit),
    ('tilt_carrier vs housing', tilt_carrier_fit, housing_fit),
    ('twist_sleeve vs housing', twist_sleeve_fit, housing_fit),
    ('plunger vs twist_sleeve', plunger_fit, twist_sleeve_fit),
    ('tilt_carrier vs twist_ring', tilt_carrier_fit, twist_ring_fit),
]

print('Quick fit-check (AABB overlap heuristic):')
for label, a, b in fit_pairs:
    clash, (dx, dy, dz) = _bbox_clash(a, b)
    state = 'CLASH-LIKELY' if clash else 'clear-or-touching'
    print(f'  - {label:28s} -> {state:16s} overlap=({dx:.2f}, {dy:.2f}, {dz:.2f}) mm')

if HAS_SHOW:
    show(housing_fit, tilt_ring_fit, twist_ring_fit, tilt_carrier_fit, twist_sleeve_fit, plunger_fit)

with BuildPart() as fit_assembly:
    insert(housing_fit)
    insert(tilt_ring_fit)
    insert(twist_ring_fit)
    insert(tilt_carrier_fit)
    insert(twist_sleeve_fit)
    insert(plunger_fit)

export_step(fit_assembly.part, 'joystick_fitcheck_assembly.step')
print('Exported: joystick_fitcheck_assembly.step')

Quick fit-check (AABB overlap heuristic):
  - tilt_ring vs housing         -> CLASH-LIKELY     overlap=(41.00, 42.00, 5.40) mm
  - twist_ring vs housing        -> CLASH-LIKELY     overlap=(32.00, 32.00, 5.80) mm
  - tilt_carrier vs housing      -> CLASH-LIKELY     overlap=(56.00, 56.00, 24.00) mm
  - twist_sleeve vs housing      -> CLASH-LIKELY     overlap=(27.50, 20.00, 30.00) mm
  - plunger vs twist_sleeve      -> CLASH-LIKELY     overlap=(7.20, 7.20, 42.00) mm
  - tilt_carrier vs twist_ring   -> clear-or-touching overlap=(32.00, 32.00, -1.00) mm
++++++
Exported: joystick_fitcheck_assembly.step


## Step 5 Applied: Auto-Tuning of Stackup and Bolt Circles

This step runs a small search over:
- Z offsets for ring and moving subassemblies
- Candidate M2.5 bolt-circle diameters for tilt and twist stop rings

It chooses a best candidate score from:
- Fewer AABB clash flags
- Lower overlap-volume heuristic
- Valid radial edge margins for hole placement on both rings

In [15]:
from itertools import product

def _hole_margin_penalty(ring_od, ring_id, bcd, hole_d, min_margin=1.2):
    outer_r = ring_od / 2
    inner_r = ring_id / 2
    center_r = bcd / 2
    hole_r = hole_d / 2
    inner_margin = center_r - hole_r - inner_r
    outer_margin = outer_r - (center_r + hole_r)
    penalty = 0.0
    if inner_margin < min_margin:
        penalty += (min_margin - inner_margin) * 1000.0
    if outer_margin < min_margin:
        penalty += (min_margin - outer_margin) * 1000.0
    return penalty, inner_margin, outer_margin

def _stackup_score(cfg):
    z_tilt_ring, z_twist_ring, z_tilt_carrier, z_twist_sleeve, z_plunger, bcd_tilt, bcd_twist = cfg

    # Reuse the existing AABB heuristic from Step 4.
    h = housing_m25.moved(Location((0, 0, 0)))
    tr = tilt_stop_ring_m25.moved(Location((0, 0, z_tilt_ring)))
    wr = twist_stop_ring_m25.moved(Location((0, 0, z_twist_ring)))
    tc = tilt_carrier.moved(Location((0, 0, z_tilt_carrier)))
    ws = twist_sleeve.moved(Location((0, 0, z_twist_sleeve)))
    pl = plunger.moved(Location((0, 0, z_plunger)))

    pairs = [
        (tr, h), (wr, h), (tc, h), (ws, h), (pl, ws), (tc, wr)
    ]

    clash_count = 0
    overlap_volume_heur = 0.0
    for a, b in pairs:
        clash, (dx, dy, dz) = _bbox_clash(a, b)
        if clash:
            clash_count += 1
            overlap_volume_heur += max(dx, 0) * max(dy, 0) * max(dz, 0)

    # Ring geometry from Step 2 functions (matches current param values).
    tilt_pen, tilt_inner_m, tilt_outer_m = _hole_margin_penalty(
        ring_od=40.0, ring_id=P['shaft_od'] + 1.0, bcd=bcd_tilt, hole_d=P['m25_pass_d']
    )
    twist_pen, twist_inner_m, twist_outer_m = _hole_margin_penalty(
        ring_od=32.0, ring_id=P['shaft_od'] + 0.8, bcd=bcd_twist, hole_d=P['m25_pass_d']
    )

    total_score = clash_count * 1_000_000.0 + overlap_volume_heur + tilt_pen + twist_pen
    meta = {
        'clash_count': clash_count,
        'overlap_volume_heur': overlap_volume_heur,
        'tilt_inner_margin': tilt_inner_m,
        'tilt_outer_margin': tilt_outer_m,
        'twist_inner_margin': twist_inner_m,
        'twist_outer_margin': twist_outer_m,
    }
    return total_score, meta

z_tilt_ring_vals = [-34.0, -33.0, -32.0]
z_twist_ring_vals = [-30.0, -29.0, -28.0]
z_tilt_carrier_vals = [-13.0, -11.0, -9.0]
z_twist_sleeve_vals = [-4.0, -2.0, 0.0]
z_plunger_vals = [-22.0, -20.0, -18.0, -16.0]
bcd_tilt_vals = [26.0, 28.0, 30.0]
bcd_twist_vals = [20.0, 22.0, 24.0]

candidates = []
for cfg in product(
    z_tilt_ring_vals, z_twist_ring_vals, z_tilt_carrier_vals, z_twist_sleeve_vals,
    z_plunger_vals, bcd_tilt_vals, bcd_twist_vals,
):
    score, meta = _stackup_score(cfg)
    candidates.append((score, cfg, meta))

candidates.sort(key=lambda x: x[0])
best_score, best_cfg, best_meta = candidates[0]

print('Best candidate score:', round(best_score, 2))
print('Best config:')
print('  z_tilt_ring    =', best_cfg[0])
print('  z_twist_ring   =', best_cfg[1])
print('  z_tilt_carrier =', best_cfg[2])
print('  z_twist_sleeve =', best_cfg[3])
print('  z_plunger      =', best_cfg[4])
print('  bcd_tilt       =', best_cfg[5])
print('  bcd_twist      =', best_cfg[6])
print('Heuristic detail:', best_meta)

print('\nTop 5 candidates:')
for i, (score, cfg, meta) in enumerate(candidates[:5], start=1):
    print(
        f"  {i}. score={score:.1f} clashes={meta['clash_count']} ov={meta['overlap_volume_heur']:.1f} "
        f"z=({cfg[0]}, {cfg[1]}, {cfg[2]}, {cfg[3]}, {cfg[4]}) bcd=({cfg[5]}, {cfg[6]})"
    )

# Build tuned geometry from best candidate.
best_z_tilt_ring, best_z_twist_ring, best_z_tilt_carrier, best_z_twist_sleeve, best_z_plunger, best_bcd_tilt, best_bcd_twist = best_cfg

tilt_stop_ring_tuned = add_m25_bolt_pattern_to_ring(make_tilt_stop_ring(), bolt_circle_d=best_bcd_tilt)
twist_stop_ring_tuned = add_m25_bolt_pattern_to_ring(make_twist_stop_ring(), bolt_circle_d=best_bcd_twist)

housing_tuned_fit = housing_m25.moved(Location((0, 0, 0)))
tilt_ring_tuned_fit = tilt_stop_ring_tuned.moved(Location((0, 0, best_z_tilt_ring)))
twist_ring_tuned_fit = twist_stop_ring_tuned.moved(Location((0, 0, best_z_twist_ring)))
tilt_carrier_tuned_fit = tilt_carrier.moved(Location((0, 0, best_z_tilt_carrier)))
twist_sleeve_tuned_fit = twist_sleeve.moved(Location((0, 0, best_z_twist_sleeve)))
plunger_tuned_fit = plunger.moved(Location((0, 0, best_z_plunger)))

if HAS_SHOW:
    show(
        housing_tuned_fit, tilt_ring_tuned_fit, twist_ring_tuned_fit,
        tilt_carrier_tuned_fit, twist_sleeve_tuned_fit, plunger_tuned_fit
    )

with BuildPart() as tuned_fit_assembly:
    insert(housing_tuned_fit)
    insert(tilt_ring_tuned_fit)
    insert(twist_ring_tuned_fit)
    insert(tilt_carrier_tuned_fit)
    insert(twist_sleeve_tuned_fit)
    insert(plunger_tuned_fit)

export_step(tilt_stop_ring_tuned, 'joystick_tilt_stop_ring_m25_tuned.step')
export_step(twist_stop_ring_tuned, 'joystick_twist_stop_ring_m25_tuned.step')
export_step(tuned_fit_assembly.part, 'joystick_fitcheck_assembly_tuned.step')
print('Exported: joystick_tilt_stop_ring_m25_tuned.step, joystick_twist_stop_ring_m25_tuned.step, joystick_fitcheck_assembly_tuned.step')

Best candidate score: 5108079.28
Best config:
  z_tilt_ring    = -34.0
  z_twist_ring   = -30.0
  z_tilt_carrier = -11.0
  z_twist_sleeve = 0.0
  z_plunger      = -16.0
  bcd_tilt       = 26.0
  bcd_twist      = 22.0
Heuristic detail: {'clash_count': 5, 'overlap_volume_heur': 108079.28116480002, 'tilt_inner_margin': 3.1500000000000004, 'tilt_outer_margin': 5.65, 'twist_inner_margin': 1.25, 'twist_outer_margin': 3.6500000000000004}

Top 5 candidates:
  1. score=5108079.3 clashes=5 ov=108079.3 z=(-34.0, -30.0, -11.0, 0.0, -16.0) bcd=(26.0, 22.0)
  2. score=5108079.3 clashes=5 ov=108079.3 z=(-34.0, -30.0, -11.0, 0.0, -16.0) bcd=(26.0, 24.0)
  3. score=5108079.3 clashes=5 ov=108079.3 z=(-34.0, -30.0, -11.0, 0.0, -16.0) bcd=(28.0, 22.0)
  4. score=5108079.3 clashes=5 ov=108079.3 z=(-34.0, -30.0, -11.0, 0.0, -16.0) bcd=(28.0, 24.0)
  5. score=5108079.3 clashes=5 ov=108079.3 z=(-34.0, -30.0, -11.0, 0.0, -16.0) bcd=(30.0, 22.0)
++++++
Exported: joystick_tilt_stop_ring_m25_tuned.step, joystick_

## Step 6 Applied: True Solid-Intersection Check

This step replaces AABB-only clash flags with geometric intersection checks on the tuned placement:
- Computes real overlap volume using boolean common.
- Reports intersection volume per critical pair.
- Exports an overlap-geometry debug STEP for visual diagnosis.

In [17]:
def _solid_common(a, b):
    # Try explicit API first, then operator fallback.
    try:
        return a.intersect(b)
    except Exception:
        return a & b

def _intersection_volume(a, b):
    try:
        common = _solid_common(a, b)
        vol = float(common.volume) if common is not None else 0.0
        if vol < 1e-6:
            vol = 0.0
        return vol, common
    except Exception:
        return 0.0, None

# Use tuned placement objects from Step 5.
solid_pairs = [
    ('tilt_ring vs housing', tilt_ring_tuned_fit, housing_tuned_fit),
    ('twist_ring vs housing', twist_ring_tuned_fit, housing_tuned_fit),
    ('tilt_carrier vs housing', tilt_carrier_tuned_fit, housing_tuned_fit),
    ('twist_sleeve vs housing', twist_sleeve_tuned_fit, housing_tuned_fit),
    ('plunger vs twist_sleeve', plunger_tuned_fit, twist_sleeve_tuned_fit),
    ('tilt_carrier vs twist_ring', tilt_carrier_tuned_fit, twist_ring_tuned_fit),
]

print('True solid-intersection volumes (mm^3):')
intersection_results = []
for label, a, b in solid_pairs:
    vol, common = _intersection_volume(a, b)
    intersection_results.append((label, vol, common))
    state = 'INTERSECT' if vol > 0 else 'clear-or-touching'
    print(f'  - {label:28s} -> {state:16s} volume={vol:.4f}')

nonzero_overlaps = [(label, vol, common) for label, vol, common in intersection_results if vol > 0 and common is not None]

if nonzero_overlaps:
    with BuildPart() as overlap_debug:
        for _, _, common in nonzero_overlaps:
            insert(common)
    export_step(overlap_debug.part, 'joystick_overlap_debug.step')
    print('Exported: joystick_overlap_debug.step')
    if HAS_SHOW:
        show(overlap_debug.part)
else:
    print('No positive-volume intersections found; no overlap debug STEP exported.')

True solid-intersection volumes (mm^3):
  - tilt_ring vs housing         -> clear-or-touching volume=0.0000
  - twist_ring vs housing        -> clear-or-touching volume=0.0000
  - tilt_carrier vs housing      -> clear-or-touching volume=0.0000
  - twist_sleeve vs housing      -> clear-or-touching volume=0.0000
  - plunger vs twist_sleeve      -> clear-or-touching volume=0.0000
  - tilt_carrier vs twist_ring   -> clear-or-touching volume=0.0000
No positive-volume intersections found; no overlap debug STEP exported.


## Step 7 Applied: Export Printable Parts to joystickv2

This cell exports each printable part as a separate STEP file under:
`/Volumes/SSD 990 2TB/Documents/stl/joystickv2`

It prefers tuned/newer variants when available and falls back to baseline parts otherwise.

In [18]:
from pathlib import Path

OUT_DIR = Path('/Volumes/SSD 990 2TB/Documents/stl/joystickv2')
OUT_DIR.mkdir(parents=True, exist_ok=True)

def pick_part(*names):
    for n in names:
        if n in globals() and globals()[n] is not None:
            return globals()[n], n
    return None, None

# Preferred export map: tuned/newer first, then fallback.
part_map = {
    'joystick_housing': ('housing_m25', 'housing'),
    'joystick_tilt_carrier': ('tilt_carrier',),
    'joystick_twist_sleeve': ('twist_sleeve',),
    'joystick_knob': ('knob',),
    'joystick_plunger': ('plunger',),
    'joystick_xy_switch_bracket': ('xy_switch_bracket',),
    'joystick_twist_switch_bracket': ('twist_switch_bracket',),
    'joystick_center_switch_mount': ('center_switch_mount',),
    'joystick_tilt_stop_ring': ('tilt_stop_ring_tuned', 'tilt_stop_ring_m25', 'tilt_stop_ring'),
    'joystick_twist_stop_ring': ('twist_stop_ring_tuned', 'twist_stop_ring_m25', 'twist_stop_ring'),
}

exported = []
skipped = []
for stem, candidates in part_map.items():
    part, chosen = pick_part(*candidates)
    if part is None:
        skipped.append((stem, candidates))
        continue
    out_path = OUT_DIR / f'{stem}.step'
    export_step(part, str(out_path))
    exported.append((stem, chosen, str(out_path)))

print(f'Export directory: {OUT_DIR}')
print('Exported files:')
for stem, chosen, out_path in exported:
    print(f'  - {stem}.step  (from variable: {chosen})')

if skipped:
    print('Skipped (missing variables):')
    for stem, candidates in skipped:
        print(f'  - {stem}: tried {candidates}')

Export directory: /Volumes/SSD 990 2TB/Documents/stl/joystickv2
Exported files:
  - joystick_housing.step  (from variable: housing_m25)
  - joystick_tilt_carrier.step  (from variable: tilt_carrier)
  - joystick_twist_sleeve.step  (from variable: twist_sleeve)
  - joystick_knob.step  (from variable: knob)
  - joystick_plunger.step  (from variable: plunger)
  - joystick_xy_switch_bracket.step  (from variable: xy_switch_bracket)
  - joystick_twist_switch_bracket.step  (from variable: twist_switch_bracket)
  - joystick_center_switch_mount.step  (from variable: center_switch_mount)
  - joystick_tilt_stop_ring.step  (from variable: tilt_stop_ring_tuned)
  - joystick_twist_stop_ring.step  (from variable: twist_stop_ring_tuned)


## Step 8 Applied: Exploded Parts Diagram

This view spaces the printable parts so you can inspect how they stack and relate in assembly.

It also exports a STEP of the exploded layout for external viewing.

In [19]:
from pathlib import Path

def _pick_part(*names):
    for n in names:
        if n in globals() and globals()[n] is not None:
            return globals()[n], n
    return None, None

# Prefer tuned/newer variants where available.
part_sources = [
    ('housing', ('housing_m25', 'housing')),
    ('tilt_stop_ring', ('tilt_stop_ring_tuned', 'tilt_stop_ring_m25', 'tilt_stop_ring')),
    ('twist_stop_ring', ('twist_stop_ring_tuned', 'twist_stop_ring_m25', 'twist_stop_ring')),
    ('tilt_carrier', ('tilt_carrier',)),
    ('twist_sleeve', ('twist_sleeve',)),
    ('plunger', ('plunger',)),
    ('knob', ('knob',)),
    ('xy_switch_bracket', ('xy_switch_bracket',)),
    ('twist_switch_bracket', ('twist_switch_bracket',)),
    ('center_switch_mount', ('center_switch_mount',)),
]

# Exploded placement offsets (mm) for readability.
placements = {
    'housing': (0, 0, 0),
    'tilt_stop_ring': (0, 0, 14),
    'twist_stop_ring': (0, 0, 28),
    'tilt_carrier': (0, 0, 45),
    'twist_sleeve': (0, 0, 62),
    'plunger': (0, 0, 79),
    'knob': (0, 0, 97),
    'xy_switch_bracket': (-36, -20, 28),
    'twist_switch_bracket': (36, -20, 34),
    'center_switch_mount': (0, 34, 22),
}

exploded_parts = []
used = []
missing = []

for logical_name, candidates in part_sources:
    part, chosen_name = _pick_part(*candidates)
    if part is None:
        missing.append((logical_name, candidates))
        continue
    moved = part.moved(Location(placements[logical_name]))
    exploded_parts.append(moved)
    used.append((logical_name, chosen_name, placements[logical_name]))

print('Exploded view parts used:')
for logical_name, chosen_name, pos in used:
    print(f'  - {logical_name:20s} <- {chosen_name:24s} at {pos}')

if missing:
    print('Missing parts (not shown):')
    for logical_name, candidates in missing:
        print(f'  - {logical_name}: tried {candidates}')

if HAS_SHOW and exploded_parts:
    show(*exploded_parts)

if exploded_parts:
    with BuildPart() as exploded_assembly:
        for p in exploded_parts:
            insert(p)

    out_dir = Path('/Volumes/SSD 990 2TB/Documents/stl/joystickv2')
    out_dir.mkdir(parents=True, exist_ok=True)
    out_file = out_dir / 'joystick_exploded_assembly.step'
    export_step(exploded_assembly.part, str(out_file))
    print(f'Exported: {out_file}')
else:
    print('No parts available to create exploded assembly.')

Exploded view parts used:
  - housing              <- housing_m25              at (0, 0, 0)
  - tilt_stop_ring       <- tilt_stop_ring_tuned     at (0, 0, 14)
  - twist_stop_ring      <- twist_stop_ring_tuned    at (0, 0, 28)
  - tilt_carrier         <- tilt_carrier             at (0, 0, 45)
  - twist_sleeve         <- twist_sleeve             at (0, 0, 62)
  - plunger              <- plunger                  at (0, 0, 79)
  - knob                 <- knob                     at (0, 0, 97)
  - xy_switch_bracket    <- xy_switch_bracket        at (-36, -20, 28)
  - twist_switch_bracket <- twist_switch_bracket     at (36, -20, 34)
  - center_switch_mount  <- center_switch_mount      at (0, 34, 22)
++++++++++
Exported: /Volumes/SSD 990 2TB/Documents/stl/joystickv2/joystick_exploded_assembly.step


## Step 9 Applied: Housing Fit Fix (Serviceable Assembly)

This patch replaces the housing with an assembly-friendly version:
- Corrects the shaft bore to cut through full housing depth.
- Adds a large bottom service opening for part insertion.
- Adds side windows so switch brackets can be installed/serviced.
- Rebuilds the M2.5 boss variant from the corrected housing.

In [ ]:
def make_housing_v2(p=P):
    plate_xy = p['plate_xy']
    plate_t = p['plate_t']
    body_xy = p['body_xy']
    body_z = p['body_z']

    wall = 3.0
    inner_xy = body_xy - 2.0 * wall
    bottom_open_xy = inner_xy - 4.0

    with BuildPart() as housing_v2:
        # Top plate (panel side)
        with BuildSketch(Plane.XY):
            Rectangle(plate_xy, plate_xy)
        extrude(amount=plate_t)

        # Under-panel body volume
        with BuildSketch(Plane.XY.offset(-body_z)):
            Rectangle(body_xy, body_xy)
        extrude(amount=body_z)

        # Hollow inner cavity (keeps ~3 mm walls)
        with BuildSketch(Plane.XY.offset(-body_z + wall)):
            Rectangle(inner_xy, inner_xy)
        extrude(amount=body_z - wall, mode=Mode.SUBTRACT)

        # Full-depth shaft bore (fixes prior incorrect one-sided cut)
        with BuildSketch(Plane.XY.offset(-body_z)):
            Circle((p['shaft_od'] / 2) + 1.0)
        extrude(amount=body_z + plate_t, mode=Mode.SUBTRACT)

        # Bottom service opening for assembly insertion
        with BuildSketch(Plane.XY.offset(-body_z)):
            Rectangle(bottom_open_xy, bottom_open_xy)
        extrude(amount=wall + 0.2, mode=Mode.SUBTRACT)

        # Side service windows for switch bracket installation
        win_w = 18.0
        win_h = 12.0
        win_z = -body_z + 14.0
        for sx in (-1, 1):
            with Locations((sx * (body_xy / 2), 0, win_z)):
                Box(3.6, win_w, win_h, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.MIN))
        for sy in (-1, 1):
            with Locations((0, sy * (body_xy / 2), win_z)):
                Box(win_w, 3.6, win_h, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.MIN))

        # Cover-hole pass-throughs around bottom opening
        hole_off = (bottom_open_xy / 2) - 4.5
        for sx in (-1, 1):
            for sy in (-1, 1):
                with Locations((sx * hole_off, sy * hole_off, -body_z + 0.2)):
                    Cylinder(radius=p['m25_pass_d'] / 2, height=6.0, mode=Mode.SUBTRACT, align=(Align.CENTER, Align.CENTER, Align.MIN))

    return housing_v2.part

def make_housing_with_stop_insert_bosses_v2(p=P):
    base = make_housing_v2(p)

    boss_radius = 3.2
    boss_height = 7.0
    boss_circle_d = 28.0
    boss_z = -p['body_z'] + 4.0

    with BuildPart() as enriched:
        insert(base)

        for i in range(4):
            a = math.radians(i * 90.0 + 45.0)
            x = (boss_circle_d / 2) * math.cos(a)
            y = (boss_circle_d / 2) * math.sin(a)

            with Locations((x, y, boss_z)):
                Cylinder(radius=boss_radius, height=boss_height, align=(Align.CENTER, Align.CENTER, Align.MIN))

            with Locations((x, y, boss_z)):
                Cylinder(
                    radius=p['insert_hole_d'] / 2,
                    height=p['insert_hole_depth'],
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER, Align.MIN),
                )

            with Locations((x, y, boss_z + boss_height)):
                Cylinder(
                    radius=p['m25_pass_d'] / 2,
                    height=12.0,
                    mode=Mode.SUBTRACT,
                    align=(Align.CENTER, Align.CENTER, Align.MIN),
                )

    return enriched.part

# Override active housing variables with assembly-friendly geometry.
housing = make_housing_v2()
housing_m25 = make_housing_with_stop_insert_bosses_v2()

# Quick insertion-capacity check against key largest parts.
opening_xy = (P['body_xy'] - 2.0 * 3.0) - 4.0
bb_tilt = tilt_carrier.bounding_box()
bb_ring = tilt_stop_ring_tuned.bounding_box() if 'tilt_stop_ring_tuned' in globals() else tilt_stop_ring.bounding_box()
tilt_span = max(bb_tilt.size.X, bb_tilt.size.Y)
ring_span = max(bb_ring.size.X, bb_ring.size.Y)

print(f'Bottom opening clear span: {opening_xy:.2f} mm')
print(f'Tilt carrier max span:     {tilt_span:.2f} mm')
print(f'Tilt ring max span:        {ring_span:.2f} mm')
print('Insertion check (span only):')
print('  - tilt carrier fits:' , tilt_span < opening_xy)
print('  - tilt stop ring fits:', ring_span < opening_xy)

if HAS_SHOW:
    show(housing_m25)

export_step(housing_m25, 'joystick_housing_m25.step')
print('Exported updated housing: joystick_housing_m25.step')